In [9]:
import pandas as pd
import time
import numpy as np
import requests
from bs4 import BeautifulSoup

In [3]:
phenos = pd.read_csv("../data/processed/philly_species_pheno_DOY.csv")

In [10]:

def scrape_type(url, session, timeout=15):
    """
    Fetch a MoBot PlantFinder page and return (value, status).
    status is 'ok' or a failure reason. value is the scraped string
    or a sentinel describing what went wrong.
    """
    if pd.isna(url) or not str(url).strip():
        return "NO_URL", "no_url"

    try:
        resp = session.get(str(url).strip(), timeout=timeout)
        resp.raise_for_status()
    except requests.RequestException as e:
        return f"FETCH_ERROR: {type(e).__name__}", "fetch_error"

    soup = BeautifulSoup(resp.text, "html.parser")
    type_div = soup.find("div", id="MainContentPlaceHolder_TypeRow")
    if type_div is None:
        return "NO_TYPE_ROW", "no_type_row"

    text = type_div.get_text(strip=True)  # "Type: Needled evergreen"
    if ":" not in text:
        return "UNPARSEABLE", "unparseable"

    value = text.split(":", 1)[1].strip()
    if not value:
        return "EMPTY_VALUE", "empty_value"

    return value, "ok"







In [11]:
def add_type_column(df, url_col="Detail URL", delay=1.0):
    """
    Add a 'Type' column by scraping each unique Detail URL once.
    Returns (df, failures_df). All scraped/sentinel values stay in
    the Type column; failures_df logs every non-'ok' outcome.
    """
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (research data collection)"})

    cache = {}  # url key -> (value, status)
    types = []
    failures = []

    for idx, url in df[url_col].items():
        key = str(url).strip() if pd.notna(url) else None
        if key not in cache:
            cache[key] = scrape_type(url, session)
            time.sleep(delay)  # be polite to the server

        value, status = cache[key]
        types.append(value)
        if status != "ok":
            failures.append(
                {"index": idx, "url": url, "status": status, "value": value}
            )

    df["Type"] = types
    failures_df = pd.DataFrame(
        failures, columns=["index", "url", "status", "value"]
    )
    return df, failures_df

In [12]:
phenos, type_failures = add_type_column(phenos)

In [13]:
phenos

,tree_name,scientific_name,common_name,Genus,Species,plant_code,Morphology/Physiology | Fall Conspicuous,Morphology/Physiology | Flower Color,Morphology/Physiology | Flower Conspicuous,Morphology/Physiology | Foliage Color,...,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL,bloom_range_DOY,fruit_range_DOY,Type
0,Abies balsamea - balsam fir,abies balsamea,balsam fir,Abies,balsamea,ABBA,No,Yellow,No,Green,...,abies balsamea,Non-flowering,Non-flowering,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...,NaN,"[244, 245, 246, 247, 248, 249, 250, 251, 252, ...",Needled evergreen
1,Abies fraseri - fraser fir,abies fraseri,fraser fir,Abies,fraseri,ABFR,No,Purple,No,Dark Green,...,abies fraseri,Non-flowering,N/a,N/a,"Fragrant, Evergreen",matched,https://www.missouribotanicalgarden.org/PlantF...,NaN,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 325, ...",Needled evergreen
2,Acer ginnala - amur maple,acer ginnala,amur maple,Acer,ginnala,ACGI,Yes,White,No,Green,...,acer ginnala,N/a,N/a,N/a,N/a,no results container,NaN,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 172, ...","[152, 153, 154, 155, 156, 157, 158, 159, 160, ...",NO_URL
3,Acer negundo - boxelder,acer negundo,boxelder,Acer,negundo,ACNE2,Yes,White,No,Green,...,acer negundo,March to April,Greenish-yellow,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 7...","[152, 153, 154, 155, 156, 157, 158, 159, 160, ...",Tree
4,Acer nigrum - black maple,acer nigrum,black maple,Acer,nigrum,ACNI5,Yes,Yellow,No,Green,...,acer nigrum,N/a,N/a,N/a,N/a,no results container,NaN,"[91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101,...","[152, 153, 154, 155, 156, 157, 158, 159, 160, ...",NO_URL
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289,Malus species - sugar tyme crabapple,malus species,sugar tyme crabapple,Malus,species,NaN,NaN,NaN,NaN,NaN,...,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN,NO_URL
290,Rosa species - other rose,rosa species,other rose,Rosa,species,NaN,NaN,NaN,NaN,NaN,...,rosa species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN,NO_URL
291,Acer henryii – henrys maple,acer henryii,henrys maple,Acer,henryii,NaN,NaN,NaN,NaN,NaN,...,acer henryii,N/a,N/a,N/a,N/a,no results container,NaN,NaN,NaN,NO_URL
292,Malus species - indian summer crabapple,malus species,indian summer crabapple,Malus,species,NaN,NaN,NaN,NaN,NaN,...,malus species,N/a,N/a,N/a,N/a,no exact match,NaN,NaN,NaN,NO_URL


In [14]:
phenos.groupby("Type").size()

Type
Broadleaf evergreen      2
Deciduous shrub         12
Fruit                    5
NO_URL                  77
Needled evergreen       19
Tree                   179
dtype: int64

In [18]:
phenos.loc[phenos["Type"] == "Tree"]

,tree_name,scientific_name,common_name,Genus,Species,plant_code,Morphology/Physiology | Fall Conspicuous,Morphology/Physiology | Flower Color,Morphology/Physiology | Flower Conspicuous,Morphology/Physiology | Foliage Color,...,Scientific Name,Bloom Time,Bloom Description,Flower,Leaf,Status,Detail URL,bloom_range_DOY,fruit_range_DOY,Type
3,Acer negundo - boxelder,acer negundo,boxelder,Acer,negundo,ACNE2,Yes,White,No,Green,...,acer negundo,March to April,Greenish-yellow,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 7...","[152, 153, 154, 155, 156, 157, 158, 159, 160, ...",Tree
6,Acer platanoides - crimson king norway maple,acer platanoides,crimson king norway maple,Acer,platanoides,ACPL,Yes,Green,No,Green,...,acer platanoides,March to April,Yellow,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 7...","[152, 153, 154, 155, 156, 157, 158, 159, 160, ...",Tree
7,Acer platanoides - norway maple,acer platanoides,norway maple,Acer,platanoides,ACPL,Yes,Green,No,Green,...,acer platanoides,March to April,Yellow,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 7...","[152, 153, 154, 155, 156, 157, 158, 159, 160, ...",Tree
8,Acer pseudoplatanus - sycamore maple,acer pseudoplatanus,sycamore maple,Acer,pseudoplatanus,ACPS,No,Green,No,Yellow-Green,...,acer pseudoplatanus,May,Yellow green,Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...,"[121, 122, 123, 124, 125, 126, 127, 128, 129, ...","[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 234, ...",Tree
9,Acer rubrum - red maple,acer rubrum,red maple,Acer,rubrum,ACRU,Yes,Red,Yes,Green,...,acer rubrum,March to April,"Red, sometimes yellow",Showy,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 7...","[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 7...",Tree
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,Acer truncatum - purpleblow maple,acer truncatum,purpleblow maple,Acer,truncatum,NaN,NaN,NaN,NaN,NaN,...,acer truncatum,April,Greenish yellow,Insignificant,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...,"[91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101,...",NaN,Tree
278,Prunus subhirtella - snow goose cherry,prunus subhirtella,snow goose cherry,Prunus,subhirtella,NaN,NaN,NaN,NaN,NaN,...,prunus subhirtella,April,Pink to white,Showy,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...,"[91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101,...",NaN,Tree
284,Acer triflorum – three flowered maple,acer triflorum,three flowered maple,Acer,triflorum,NaN,NaN,NaN,NaN,NaN,...,acer triflorum,April,Greenish yellow,Insignificant,Good Fall,matched,https://www.missouribotanicalgarden.org/PlantF...,"[91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101,...",NaN,Tree
285,Carpinus betulus - fastigiate hornbeam,carpinus betulus,fastigiate hornbeam,Carpinus,betulus,NaN,NaN,NaN,NaN,NaN,...,carpinus betulus,March,Yellow (male) and green (female),Insignificant,N/a,matched,https://www.missouribotanicalgarden.org/PlantF...,"[60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 7...",NaN,Tree


In [19]:
phenos.to_csv("/Users/prince/philly-tree-mapper/data/processed/philly_species_pheno_DOY.csv")